# 01 · Classificação de Carteiras

Este notebook treina e avalia os **modelos de classificação binária** para o problema "esta carteira é boa (Sharpe ≥ mediana) ou ruim (Sharpe < mediana)?"

**Algoritmos baseline (hiperparâmetros padrão):**
- KNN
- Árvore de Decisão
- Random Forest
- Regressão Logística

**Modelo de fine-tuning:**
- XGBoost com GridSearchCV

---

## Setup

In [ ]:
import sys
from pathlib import Path

# Permite importar o pacote src/ a partir do notebook
sys.path.insert(0, str(Path.cwd().parents[0]))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

In [ ]:
from src.data.loader import load_carteiras_ml
from src.data.splitter import tabular_split
from src.features.portfolio_features import build_classification_dataset
from src.models.classifiers import get_classifier, CLASSIFIER_REGISTRY
from src.models.xgboost_model import XGBoostModel
from src.evaluation.classification_metrics import evaluate_classifier, print_classification_report
from src.evaluation.model_comparator import compare_classifiers
from src.visualization.classification_plots import (
    plot_confusion_matrix,
    plot_roc_curves,
    plot_metrics_comparison,
)
from src.analysis.feature_importance import get_feature_importance, plot_feature_importance

## 1. Carregamento dos Dados

In [ ]:
df = load_carteiras_ml()
df.head()

In [ ]:
X, y = build_classification_dataset(df, target='sharpe_label')
print('Features:', list(X.columns))
print('Target distribution:')
print(y.value_counts(normalize=True).round(3))

## 2. Split Treino/Teste

Split estratificado 80/20 — preserva a proporção de classes em ambos os conjuntos.

In [ ]:
X_train, X_test, y_train, y_test = tabular_split(X, y, test_size=0.20, stratify=True)

## 3. Padronização

KNN e Regressão Logística são **sensíveis à escala**. Aplicamos `StandardScaler` apenas para esses modelos. Árvores não precisam.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

SCALE_SENSITIVE = {'knn', 'logistic_regression'}

## 4. Treinamento dos Modelos Baseline

In [ ]:
baseline_results = {}
baseline_models = {}

for name in CLASSIFIER_REGISTRY:
    print(f'\n━━━ {name} ━━━')
    clf = get_classifier(name)

    if name in SCALE_SENSITIVE:
        X_tr, X_te = X_train_scaled, X_test_scaled
    else:
        X_tr, X_te = X_train.values, X_test.values

    clf.fit(X_tr, y_train)
    y_pred = clf.predict(X_te)
    y_proba = clf.predict_proba(X_te)

    metrics = evaluate_classifier(y_test, y_pred, y_proba=y_proba)
    baseline_results[name] = metrics
    baseline_models[name]  = clf

    print_classification_report(metrics, model_name=name)

## 5. XGBoost — Modelo com Fine-Tuning

In [ ]:
# Grid reduzido aqui pra rodar mais rápido no notebook;
# o pipeline em pipelines/train_xgboost.py usa o grid completo do YAML
param_grid_quick = {
    'n_estimators':  [100, 300],
    'max_depth':     [3, 5],
    'learning_rate': [0.05, 0.1],
}

xgb_clf = XGBoostModel(
    task='classification',
    param_grid=param_grid_quick,
    cv_folds=5,
    scoring='f1',
)
xgb_clf.fit(X_train, y_train)

In [ ]:
print('Melhores parâmetros encontrados:')
for k, v in xgb_clf.best_params_.items():
    print(f'  {k}: {v}')
print(f'\nMelhor F1 em CV: {xgb_clf.best_score_:.4f}')

In [ ]:
y_pred_xgb  = xgb_clf.predict(X_test)
y_proba_xgb = xgb_clf.predict_proba(X_test)

metrics_xgb = evaluate_classifier(y_test, y_pred_xgb, y_proba=y_proba_xgb)
baseline_results['xgboost'] = metrics_xgb
baseline_models['xgboost']  = xgb_clf

print_classification_report(metrics_xgb, model_name='XGBoost (Fine-Tuned)')

## 6. Comparação Final dos Modelos

In [ ]:
df_compare = compare_classifiers(baseline_results)
df_compare

In [ ]:
plot_metrics_comparison(df_compare, save_as='comparacao_classificadores.png')

### Matrizes de Confusão

In [ ]:
for name, metrics in baseline_results.items():
    cm = np.array(metrics['confusion_matrix'])
    plot_confusion_matrix(
        cm,
        title=f'Matriz de Confusão — {name}',
        save_as=f'cm_{name}.png',
    )

### Curvas ROC

In [ ]:
# Para o ROC precisamos passar X_test no formato apropriado pra cada modelo
# Como modelos sensíveis a escala foram treinados em dados padronizados,
# vamos plotar separadamente os modelos não-sensíveis a escala
non_sensitive = {name: clf.model for name, clf in baseline_models.items()
                 if name not in SCALE_SENSITIVE}

plot_roc_curves(non_sensitive, X_test, y_test, save_as='roc_curves.png')

## 7. Importância de Features

In [ ]:
# Random Forest e XGBoost expõem feature_importances_
for name in ['random_forest', 'xgboost']:
    print(f'\n━━━ {name} ━━━')
    df_imp = get_feature_importance(baseline_models[name].model, list(X.columns))
    print(df_imp)
    plot_feature_importance(
        df_imp,
        title=f'Feature Importance — {name}',
        save_as=f'feat_imp_{name}.png',
    )

## 8. Conclusões

Esta seção é onde você vai responder, no contexto do edital:

1. **Qual modelo teve melhor desempenho?** → consultar `df_compare` e a coluna F1
2. **Por quê?** → comparar a natureza dos modelos (linear vs árvore vs ensemble)
3. **Os dados influenciaram no resultado?** → balanceamento, separabilidade linear, ruído

O XGBoost (com fine-tuning) deve aparecer no topo, mas o Random Forest baseline costuma ficar muito próximo, mostrando que para esse problema a complexidade adicional do fine-tuning trouxe ganho marginal — um achado relevante para a discussão.